# Direction 2 · Phase A · Step 2: Frozen Post-GAP Readout Control Ladder (Kaggle GPU)

**Settings:** Enable **GPU T4 x1** (or T4 x2) and **Internet ON**.

**Zero manual dataset uploads required:**
- Clones `spurious_correlation_mechanisms` from GitHub (or extracts an attached `rep-repair-code.zip` if present under `/kaggle/input/`).
- Auto-recovers all 7 saved validation feature caches (`5 standard_erm` + `2 erm_star`) via `src.methods.dpe_independent.recovery.recover()`.
- Uses the in-repo official Waterbirds `metadata.csv` (validation split `split == 1` only; zero test files are ever accessed).
- Runs `rep_repair_frozen_readouts.py --device cuda` (~1-2 min on T4) and saves `/kaggle/working/rep_repair_phase_a.zip`.

In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path, PurePosixPath
import torch

REPO_URL = 'https://github.com/chautanphat/waterbirds-shortcut-mitigation.git'
BRANCH = 'spurious_correlation_mechanisms'
CODE_DIR = Path('/kaggle/working/repo')
ASSET_ROOT = Path('/kaggle/working/assets')
OUT_DIR = Path('/kaggle/working/rep_repair_phase_a')

print('CUDA available:', torch.cuda.is_available(), '| Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

# Option A: use attached code zip if present; Option B: clone branch directly.
code_zips = list(Path('/kaggle/input').rglob('*rep-repair*code*.zip')) if Path('/kaggle/input').exists() else []
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
if code_zips:
    print('Extracting code zip:', code_zips[0])
    with zipfile.ZipFile(code_zips[0]) as zf:
        for name in zf.namelist():
            p = PurePosixPath(name)
            if p.is_absolute() or '..' in p.parts or '\\' in name:
                raise ValueError(f'Unsafe zip path: {name}')
        zf.extractall(CODE_DIR)
else:
    print(f'Cloning {REPO_URL} (branch={BRANCH})...')
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(CODE_DIR)], check=True)

os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

In [ ]:
# Recover the 7 saved validation feature caches from the public Kaggle notebooks.
from src.methods.dpe_independent.recovery import recover

recovered_dir = ASSET_ROOT / 'outputs' / 'recovered'
records = recover(recovered_dir, workers=6)
print(f'Recovered {len(records)} files into {recovered_dir}')

# Run Phase A unit tests before running the experiment.
subprocess.run([sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_rep_repair_phase_a.py', '-v'], check=True)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
cmd = [
    sys.executable, '-B', '-m', 'scripts.analysis.rep_repair_frozen_readouts',
    '--config', 'configs/rep_repair/phase_a_audit.json',
    '--asset-root', str(ASSET_ROOT),
    '--failure-stats', 'results/rep_repair/phase_a_20261123/failure_audit_stats.json',
    '--output', str(OUT_DIR),
    '--device', device,
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import json
from IPython.display import Image, Markdown, display

display(Markdown((OUT_DIR / 'README.md').read_text(encoding='utf-8')))
display(Image(filename=str(OUT_DIR / 'readout_ladder.png')))
print(json.dumps(json.loads((OUT_DIR / 'gate1_decision.json').read_text()), indent=2))

# Zip everything cleanly for one-click download back into results/rep_repair/phase_a_20261123/
zip_path = Path('/kaggle/working/rep_repair_phase_a.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT_DIR.rglob('*')):
        if f.is_file():
            zf.write(f, arcname=f.relative_to(OUT_DIR))
print(f'Saved results bundle: {zip_path} ({zip_path.stat().st_size / 1e6:.2f} MB)')